# Lesson 01 - Sissejuhatus tehisintellekti agentidesse

Tere tulemast esimesse loengusse kursusel **Tehisintellekti agentide algajatele**!

**Tehisintellekti agent** on programm, mis kasutab suurt keelemudelit (LLM) kui oma järeldusmootorit ja suudab võtta *tegevusi* reaalmaailmas — kutsuda API-sid, pärida andmebaase või käivitada koodi — eesmärgiga saavutada kasutaja nimel mingi siht.

Selles märkmikus ehitad oma esimese agendi: **reiseagent**, kes soovitab puhkuse sihtkohti. Teekonnal õpid, kuidas:

1. Ühenduda Azure AI Foundry agenditeenusega kasutades **Microsoft Agent Frameworki**.
2. Anda agendile **tööriist** — lihtsa Pythoni funktsiooni, mida ta saab kutsuda.
3. Käivitada agent ja uurida tema vastust.
4. Voogedastada agendi vastus tokeni haaval.


## Seadistamine

Enne selle märkmiku käivitamist veenduge, et teil oleks:

1. **Azure AI Foundry projekt** koos juurutatud vestlusmudeliga (nt `gpt-4o-mini`).
2. **Sisse logitud Azure CLI abil** — käivitage terminalis `az login`.
3. **Määratud vajalikud keskkonnamuutujad:**
   - `AZURE_AI_PROJECT_ENDPOINT` — teie Azure AI Foundry projekti lõpp-punkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — teie juurutatud mudeli nimi.

Allolev lahter paigaldab vajalikud Python paketid.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Oma esimest agendi loomine

Agendil on vaja kahte asja:

- **Juhised**, mis ütlevad talle *kes ta on* ja *kuidas käituda* (süsteemi prompt).
- **Tööriistad** — Python funktsioonid, mis on kaunistatud `@tool` ja mida agent saab kutsuda informatsiooni saamiseks või toimingute tegemiseks.

Allpool defineerime lihtsa tööriista, mis tagastab populaarsete puhkusekohtade nimekirja. Agent kasutab seda tööriista, kui kasutaja palub reisisoovitusi.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Voogedastuse vastused

Veelgi interaktiivsema kogemuse saamiseks saate esindaja vastuse **voogedastada**. Selle asemel, et oodata täismahus vastust, annab esindaja tekstitükke, kui need genereeritakse. See on eriti kasulik vestlusliidestes, kus soovite kuvada väljundit reaalajas.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisid sa, kuidas:

- **Luua pakkuja**, mis ühendub Azure AI Foundry Agent teenusega kaudu `AzureAIProjectAgentProvider`.
- **Määratleda tööriist** kasutades `@tool` dekoratiivset märgendit, nii et agent saab kutsuda sinu Python funktsioone.
- **Käivitada agent** kasutajalt sõnumiga ja printida tema vastus.
- **Voogedastada vastuseid** reaalajas väljundi saamiseks.

Järgmises õppetükis uurime agentseid raamistikke põhjalikumalt ning õpime, kuidas anda agentidele võimsamaid tööriistu ja mitme sammu mõtlemisvõimeid.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
